# dataclass-training-args — ex1: TrainingArgs dataclass with __post_init__ validation + asdict round-trip

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataclass-training-args`. Running the final beacon cell reports progress against the `Config: @dataclass training args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: @dataclass training args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataclass-training-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataclass-training-args"
DD_SUBTOPIC = "Config: @dataclass training args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Config: `@dataclass` training args — quick refresher

Modern training-loop code groups all hyperparameters into a single args object instead of dragging dozens of kwargs through every function. The idiom:

```python
from dataclasses import dataclass, asdict

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
```

**Why a dataclass and not a plain dict.** Dataclasses give you type annotations (IDE autocomplete + static checking), default values, AND `__post_init__` for cheap validation. They keep the schema in one place — change a default and every call site picks it up.

**`frozen=False` is the right default for training args.** ARENA and most reference implementations leave the dataclass mutable so you can override fields from the CLI / config file *after* construction. `frozen=True` would force you to build a fresh object for every override, which clashes with the precedence-merge pattern (see `hparam-precedence-merge`).

**`dataclasses.asdict(args)`** is how you flatten the object back to a dict — required for wandb config logging and for JSON checkpoints. It's recursive: nested dataclasses get unrolled too.

### Exercise 1 — TrainingArgs dataclass with __post_init__ validation + asdict round-trip

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the `@dataclass` + `__post_init__` + `dataclasses.asdict` pattern to define a TrainingArgs container that validates its fields and round-trips to a plain dict.
> Keywords: dataclass, config, validation, asdict
> ```

**KCs targeted:** `dataclass-default-fields`, `post-init-validation-and-asdict`

Implement `ex1_make_training_args` so it returns a `TrainingArgs` dataclass instance built from the given overrides. The dataclass must have:

1. Fields with defaults: `lr: float = 1e-3`, `batch_size: int = 32`, `epochs: int = 10`, `optimizer_name: str = 'adam'`.
2. A `__post_init__` that raises `ValueError` if `lr <= 0`, if `batch_size < 1`, if `epochs < 1`, or if `optimizer_name` is not in `{'sgd', 'adam', 'adamw'}`.
3. A method `to_dict(self) -> dict` that returns `dataclasses.asdict(self)`.

`ex1_make_training_args` should:
- Accept `**overrides`.
- Construct a `TrainingArgs(**overrides)` and return it.

The test verifies the defaults, the validation, and the `to_dict()` round-trip.

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class TrainingArgs:
    # Fill in the fields and __post_init__ here.
    pass


def ex1_make_training_args(**overrides) -> 'TrainingArgs':
    """Build a TrainingArgs with the given overrides."""
    raise NotImplementedError()


def _test_ex1():
    # === Defaults ===
    args = ex1_make_training_args()
    assert args.lr == 1e-3, f'default lr should be 1e-3, got {args.lr}'
    assert args.batch_size == 32, f'default batch_size should be 32, got {args.batch_size}'
    assert args.epochs == 10, f'default epochs should be 10, got {args.epochs}'
    assert args.optimizer_name == 'adam', f'default optimizer_name should be adam, got {args.optimizer_name!r}'

    # === Overrides ===
    args2 = ex1_make_training_args(lr=3e-4, optimizer_name='sgd', epochs=5)
    assert args2.lr == 3e-4
    assert args2.optimizer_name == 'sgd'
    assert args2.epochs == 5
    assert args2.batch_size == 32, 'unset field should still hold the default'

    # === to_dict round-trip ===
    d = args2.to_dict()
    assert isinstance(d, dict), f'to_dict must return dict, got {type(d).__name__}'
    assert d == {'lr': 3e-4, 'batch_size': 32, 'epochs': 5, 'optimizer_name': 'sgd'}, (
        f'to_dict round-trip wrong: {d}'
    )
    # Rebuild from dict.
    args3 = ex1_make_training_args(**d)
    assert args3.to_dict() == d, 'rebuild-from-dict failed'

    # === Validation errors ===
    for bad in [
        dict(lr=0.0),
        dict(lr=-1e-3),
        dict(batch_size=0),
        dict(batch_size=-1),
        dict(epochs=0),
        dict(optimizer_name='rmsprop'),
        dict(optimizer_name=''),
    ]:
        try:
            ex1_make_training_args(**bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f'expected ValueError for {bad}, but no error raised')

    # === Valid optimizer names ===
    for name in ['sgd', 'adam', 'adamw']:
        a = ex1_make_training_args(optimizer_name=name)
        assert a.optimizer_name == name

    # === Dataclass is mutable (frozen=False default) ===
    args.lr = 5e-4
    assert args.lr == 5e-4, 'dataclass should be mutable for override-after-construct'

    # === asdict is recursive but flat here — just sanity check ===
    from dataclasses import asdict as _asdict
    assert _asdict(args) == args.to_dict(), 'to_dict must delegate to dataclasses.asdict'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
from dataclasses import dataclass, asdict

@dataclass
class TrainingArgs:
    lr: float = 1e-3
    batch_size: int = 32
    epochs: int = 10
    optimizer_name: str = 'adam'

    def __post_init__(self):
        if self.lr <= 0:
            raise ValueError(f'lr must be > 0, got {self.lr}')
        if self.batch_size < 1:
            raise ValueError(f'batch_size must be >= 1, got {self.batch_size}')
        if self.epochs < 1:
            raise ValueError(f'epochs must be >= 1, got {self.epochs}')
        if self.optimizer_name not in {'sgd', 'adam', 'adamw'}:
            raise ValueError(
                f'optimizer_name must be one of sgd/adam/adamw, '
                f'got {self.optimizer_name!r}'
            )

    def to_dict(self):
        return asdict(self)


def ex1_make_training_args(**overrides):
    return TrainingArgs(**overrides)
```

**Why `__post_init__` and not a regular `__init__`.** Dataclasses generate `__init__` for you. Replacing it defeats the whole point. `__post_init__` runs AFTER the generated init has assigned every field, which is exactly when you want validation to fire.

**Why `set` for the optimizer-name check.** `self.optimizer_name not in {'sgd', 'adam', 'adamw'}` is O(1) and reads like math. A list (`['sgd', 'adam', 'adamw']`) is fine for 3 items but signals you might care about order — you don't.

**`asdict` is recursive.** If you nest dataclasses (e.g. `optimizer_config: OptimizerConfig`), `asdict` unrolls the inner dataclass into a sub-dict. That's why real training scripts can dump the whole args object to JSON or to wandb in one call.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()